In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.2 MB/s eta 0:00:00


In [2]:
import os
import json
from groq import Groq
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY:")
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

Enter your GROQ_API_KEY:··········


In [3]:
def calculate(a, b, operation):
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b
    else:
        return "Unknown operation"

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform mathematical calculations and operations",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "First number"
                    },
                    "b": {
                        "type": "number",
                        "description": "Second number"
                    },
                    "operation": {
                        "type": "string",
                        "enum": [
                            "add",
                            "subtract",
                            "multiply",
                            "divide"
                        ],
                        "description": "Mathematical operations"
                    }
                },
                "required": ["a", "b", "operation"]
            }
        }
    }
]

In [5]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful mathematical assistant."
    },
    {
        "role": "user",
        "content": "What is 25 multiplied by 16?"
    }
]

In [6]:
response = client.chat.completions.create(
    model= MODEL,
    messages= messages,
    tools= tools,
    tool_choice= "auto"
)

In [7]:
message = response.choices[0].message
print(message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We just need to compute 25 * 16 = 400. Use function.', tool_calls=[ChatCompletionMessageToolCall(id='fc_750ff377-6484-4889-b5c0-b712ea2cc827', function=Function(arguments='{"a":25,"b":16,"operation":"multiply"}', name='calculate'), type='function')])


In [8]:
print(message.tool_calls)

[ChatCompletionMessageToolCall(id='fc_750ff377-6484-4889-b5c0-b712ea2cc827', function=Function(arguments='{"a":25,"b":16,"operation":"multiply"}', name='calculate'), type='function')]


In [9]:
import json

In [10]:
def calculate(a, b, operation):
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b
    return None

In [11]:
available_functions = {"calculate": calculate}

In [12]:
import json

In [13]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful mathematical assistant. "
            "Use the calculator tool whenever calculations are required."
        ),
    },
    {
        "role": "user",
        "content": "Calculate 25 * 16 and then add 100.",
    },
]

In [17]:
while True:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    assistant_message = response.choices[0].message
    messages.append(assistant_message)

    if not assistant_message.tool_calls:
        print(assistant_message.content)
        break

    # Execute every tool call
    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function = available_functions[function_name]
        result = function(**arguments)

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(result),
            }
        )

The result of \(25 \times 16 + 100\) is **500**.
